# Inspeção do ToolACE

Objetivo: entender as descrições das ferramentas, os pedidos e as respostas supervisionadas antes de preparar o treinamento.

A revisão abaixo foi identificada na inspeção inicial no Colab e fica fixada para reprodução. O registro dessa inspeção está em `reports/data_inspection.json`.

Fonte: https://huggingface.co/datasets/Team-ACE/ToolACE — licença declarada Apache 2.0.

Executar com CPU no Colab/Kaggle. As chamadas presentes nos dados são texto; este notebook não executa as ferramentas descritas.


In [ ]:
import json
from urllib.request import urlopen

toolace_revision = "6bda777c88d21e5a204703c1ee45597a8fa4f734"
url = (
    "https://huggingface.co/datasets/Team-ACE/ToolACE/"
    f"resolve/{toolace_revision}/data.json"
)
with urlopen(url, timeout=60) as resposta:
    toolace_data = json.load(resposta)

assert isinstance(toolace_data, list) and toolace_data
exemplo = toolace_data[0]
print("Revisão:", toolace_revision)
print("Exemplos:", len(toolace_data))
print("Campos:", list(exemplo))
print("\nINSTRUÇÕES E FERRAMENTAS:\n")
print(exemplo["system"])
for papel in ("user", "assistant"):
    mensagem = next(m for m in exemplo["conversations"] if m["from"] == papel)
    print(f"\n{papel.upper()}:\n{mensagem['value']}")


## Estrutura das conversas

Verificar tipos de mensagens, sequências e comprimentos antes de selecionar o recorte. Esta auditoria não avalia a correção das chamadas. Os resultados conferidos no Colab estão no relatório de inspeção.


In [ ]:
from collections import Counter
from statistics import median

papeis = Counter()
sequencias = Counter()
tamanhos = []
problemas = []

for indice, registro in enumerate(toolace_data):
    if not isinstance(registro, dict):
        problemas.append(indice)
        continue

    mensagens = registro.get("conversations")
    estrutura_valida = (
        isinstance(registro.get("system"), str)
        and isinstance(mensagens, list)
        and len(mensagens) > 0
    )
    if not estrutura_valida:
        problemas.append(indice)
        continue

    mensagens_validas = all(
        isinstance(mensagem, dict)
        and isinstance(mensagem.get("from"), str)
        and isinstance(mensagem.get("value"), str)
        for mensagem in mensagens
    )
    if not mensagens_validas:
        problemas.append(indice)
        continue

    sequencia = tuple(mensagem["from"] for mensagem in mensagens)
    papeis.update(sequencia)
    sequencias[sequencia] += 1
    tamanhos.append(len(mensagens))

print("Registros examinados:", len(toolace_data))
print("Registros com estrutura válida:", len(tamanhos))
print("Registros com problemas estruturais:", len(problemas))
print("\nQuantidade de mensagens por papel:")
for papel, quantidade in papeis.most_common():
    print(f"  {papel}: {quantidade}")
print("\nOito sequências de conversa mais frequentes:")
for sequencia, quantidade in sequencias.most_common(8):
    print(f"  {' → '.join(sequencia)}: {quantidade}")
if tamanhos:
    print("\nMensagens por conversa:")
    print("  Mínimo:", min(tamanhos))
    print("  Mediana:", median(tamanhos))
    print("  Máximo:", max(tamanhos))
if problemas:
    print("\nPrimeiros índices com problemas:", problemas[:10])
